<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Conversation Memory in AI Agents_sol </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Conversation Memory in AI Agents

**Purpose:**  
This notebook demonstrates how to implement **Conversation Memory** in AI agents, a critical component for maintaining context across multiple interactions, just like real production agent systems.

In real-world applications memory allows the agent to remember previous messages, user preferences, past decisions, and ongoing context.

We will build an **Automotive Stock Research Agent** that can answer questions about car companies (Tesla, Toyota, BMW, etc.) by scraping real-time public data and remembering context across multiple user turns.

**What You'll Learn:**
- Why agents need memory
- Conversation history storage using LangChain Memory
- Memory window management (short-term vs long-term)
- Retrieving relevant past messages
- Production considerations for memory growth and efficiency

## 1. Setup

In [1]:
# Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" faiss-cpu beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 447.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 

## 2. Tools – Real-Time Data Scraping for Car Companies

In [2]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def get_car_company_info(company: str) -> str:
    """Scrape real company overview from Wikipedia for major car manufacturers."""
    try:
        url = f"https://en.wikipedia.org/wiki/{company.replace(' ', '_')}"
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        summary = ""
        for p in soup.find_all("p")[:4]:
            text = p.get_text(strip=True)
            if len(text) > 80:
                summary += text + "\n\n"

        return f"Overview of {company}:\n\n" + summary[:900] + "...\n\n(Source: Wikipedia)"
    except Exception as e:
        return f"Could not fetch information for {company}. Error: {str(e)}"

@tool
def get_basic_stock_trend(company: str) -> str:
    """Simulate recent stock performance (in real production, this would scrape Yahoo Finance)."""
    trends = {
        "tesla": "Tesla (TSLA) has shown strong growth in 2025 with increased EV deliveries.",
        "toyota": "Toyota (TM) remains stable with strong hybrid vehicle sales.",
        "bmw": "BMW (BMWYY) has steady performance in luxury EV segment.",
        "ford": "Ford (F) is focusing on electric F-150 Lightning and commercial vehicles."
    }
    return trends.get(company.lower(), f"Recent trend for {company} is positive but market dependent.")

tools = [get_car_company_info, get_basic_stock_trend]

## 3. Why Agents Need Memory

In a real production environment, agents are rarely stateless. They interact with users over multiple turns.

**Real-world use cases:**
- A customer support agent remembering previous complaints
- A personal finance assistant recalling past investment preferences
- A research agent building upon previous findings
- User asks about Tesla → then asks for comparison with Toyota
- User mentions budget → later asks for recommendation within that budget
- User provides preferences → agent remembers them for future suggestions

Without memory, every new message is treated as a completely new conversation ,leading to repetitive questions, loss of context, and poor user experience.

Today we will implement **Conversation Memory** to solve this problem.



#4. Prompt Template

In [3]:
from langchain.prompts import PromptTemplate

unified_memory_prompt = PromptTemplate.from_template(
    """You are an expert Automotive Stock Research Assistant.
You provide clear, factual, and data-driven insights about car companies, electric vehicles, and stock performance.

Remember previous messages and build upon them logically.

Current conversation history:
{history}

Human: {input}
AI:"""
)

# Conversation Buffer Memory

## 5. Conversation Buffer Memory (Short-term Memory)

The simplest form of memory is **ConversationBufferMemory**  it stores the entire conversation history.

In [4]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_verbose, set_debug, set_llm_cache # Import all necessary globals

# Explicitly set verbosity, debug status, and LLM cache to prevent internal errors
set_verbose(True)
set_debug(False) # Set debug to False to prevent the AttributeError
set_llm_cache(None) # Explicitly set LLM cache to None to prevent the AttributeError

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

buffer_memory = ConversationBufferMemory()

buffer_conversation = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    prompt=unified_memory_prompt,
    #verbose=True
)
print("Summary Memory initialized")

Summary Memory initialized


/tmp/ipykernel_7958/4116724637.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  buffer_memory = ConversationBufferMemory()
/tmp/ipykernel_7958/4116724637.py:15: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  buffer_conversation = ConversationChain(


## Testing Short-Term Memory (Multi-turn Conversation)

In [5]:
print("Short-term Memory Test")
print(buffer_conversation.predict(input="Hi, I'm researching electric car companies. Tell me about Tesla."))
print(buffer_conversation.predict(input="How does it compare to Toyota?"))
print(buffer_conversation.predict(input="Which one has better long-term growth potential?"))#Tesla and Toyota

print("\nCurrent Short-term Memory:")
print(buffer_memory.load_memory_variables({}))

Short-term Memory Test


> Entering new ConversationChain chain...
Prompt after formatting:
You are an expert Automotive Stock Research Assistant.
You provide clear, factual, and data-driven insights about car companies, electric vehicles, and stock performance.

Remember previous messages and build upon them logically.

Current conversation history:


Human: Hi, I'm researching electric car companies. Tell me about Tesla.
AI:

> Finished chain.
Tesla is a leading electric car company founded by Elon Musk in 2003. They are known for their innovative electric vehicles, including the Model S, Model 3, Model X, and Model Y. Tesla has a strong focus on sustainability and has been a key player in the push towards electric vehicles becoming more mainstream. In terms of stock performance, Tesla has seen significant growth in recent years, with their stock price reaching new highs. They have a loyal customer base and continue to innovate in the electric vehicle space.


> Entering new Conversa

#Long-Term / Summary Memory

## 6. Long-Term Memory – ConversationSummaryMemory

In production, storing full history is expensive. Summary memory condenses old messages to save tokens.

In [6]:
from langchain.memory import ConversationSummaryMemory

summary_memory = ConversationSummaryMemory(llm=llm)

summary_conversation = ConversationChain(
    llm=llm,
    memory=summary_memory,
    prompt=unified_memory_prompt,
    verbose=True
)
print("Summary Memory initialized")


Summary Memory initialized


/tmp/ipykernel_7958/4060403967.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(llm=llm)


## Testing Summary Memory

In [7]:
print("Long-term Summary Memory Test")
print(summary_conversation.predict(input="Hi, tell me about BMW and its electric vehicles."))
print(summary_conversation.predict(input="How does BMW compare to Tesla in terms of market share?"))
print(summary_conversation.predict(input="What is the long-term outlook for BMW stock?"))

print("\nCurrent Summary Memory:")
print(summary_memory.load_memory_variables({}))

Long-term Summary Memory Test


> Entering new ConversationChain chain...
Prompt after formatting:
You are an expert Automotive Stock Research Assistant.
You provide clear, factual, and data-driven insights about car companies, electric vehicles, and stock performance.

Remember previous messages and build upon them logically.

Current conversation history:


Human: Hi, tell me about BMW and its electric vehicles.
AI:


> Entering new LLMChain chain...
Prompt after formatting:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence.

#Memory Window Management

## 5. Memory Window Management

Storing the entire conversation history forever is not scalable in production.

**Common Strategies:**
- **ConversationBufferMemory** → Stores everything (simple but grows indefinitely)
- **ConversationBufferWindowMemory** → Keeps only the last N messages (fixed window)
- **ConversationSummaryMemory** → Summarizes old messages to save tokens
- **ConversationSummaryBufferMemory** → Hybrid approach (summary + recent messages)

Let's see the window approach.

In [8]:
from langchain.memory import ConversationBufferWindowMemory

# Keep only last 4 messages (2 full turns)
window_memory = ConversationBufferWindowMemory(k=4)

window_conversation = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=True
)

print("Memory Window (k=4) initialized")

Memory Window (k=4) initialized


/tmp/ipykernel_7958/2419097143.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=4)


#Retrieval Strategies

## 6. Retrieving Relevant Past Messages

In production systems, we often need **smart retrieval** instead of dumping everything into the prompt.

We can use **ConversationSummaryMemory** or **Vector Store backed memory** for better relevance.

(ConversationSummaryMemory → Summarizes old messages to save tokens) already used above in demonstration of 'Long-Term Memory'.

In [9]:
from langchain.memory import ConversationSummaryMemory

summary_memory = ConversationSummaryMemory(llm=llm)

summary_conversation = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=True
)

print("Conversation Summary Memory initialized")

Conversation Summary Memory initialized


## 7. Testing All Memory Types with Multi-turn Conversation

In [10]:
query1 = "Hi, I'm researching electric car companies. Tell me about Tesla."
query2 = "How does Tesla compare to Toyota?"
query3 = "Which one has better long-term growth potential?"

print("=== Short-term Buffer Memory ===")
print(buffer_conversation.predict(input=query1))
print(buffer_conversation.predict(input=query2))
print(buffer_conversation.predict(input=query3))

print("\n=== Window Memory (k=4) ===")
print(window_conversation.predict(input=query1))
print(window_conversation.predict(input=query2))
print(window_conversation.predict(input=query3))

print("\n=== Summary Memory ===")
print(summary_conversation.predict(input=query1))
print(summary_conversation.predict(input=query2))
print(summary_conversation.predict(input=query3))

=== Short-term Buffer Memory ===


> Entering new ConversationChain chain...
Prompt after formatting:
You are an expert Automotive Stock Research Assistant.
You provide clear, factual, and data-driven insights about car companies, electric vehicles, and stock performance.

Remember previous messages and build upon them logically.

Current conversation history:
Human: Hi, I'm researching electric car companies. Tell me about Tesla.
AI: Tesla is a leading electric car company founded by Elon Musk in 2003. They are known for their innovative electric vehicles, including the Model S, Model 3, Model X, and Model Y. Tesla has a strong focus on sustainability and has been a key player in the push towards electric vehicles becoming more mainstream. In terms of stock performance, Tesla has seen significant growth in recent years, with their stock price reaching new highs. They have a loyal customer base and continue to innovate in the electric vehicle space.
Human: How does it compare to Toyota

## 8. Memory Growth and Pruning Strategies

**Real Production Challenges:**
- Memory can grow indefinitely, increasing token cost and slowing down the agent.
- Long conversations can exceed LLM context window limits.

**Common Pruning Strategies:**
- Fixed window (keep only last N messages)
- Summarization of old messages
- Time-based expiration
- Relevance-based pruning (keep only important messages)
- Hybrid approach (recent messages + summary of older ones)

In production systems, memory is often managed dynamically based on token count and relevance.

## Time-based Expiration Memory

In real production systems, old messages are automatically removed after a certain time to control memory size, reduce cost, and respect privacy policies.

In [11]:
from datetime import datetime, timedelta

class TimeBasedConversationMemory:
    def __init__(self, max_age_minutes: int = 5):
        self.max_age = timedelta(minutes=max_age_minutes)
        self.messages = []   # list of {"role": , "content": , "timestamp": }

    def add_message(self, role: str, content: str):
        """Add a new message and prune old ones."""
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now()
        })
        self._prune_old_messages()

    def _prune_old_messages(self):
        """Remove messages older than max_age."""
        cutoff = datetime.now() - self.max_age #go back max age mins
        original_count = len(self.messages)
        self.messages = [msg for msg in self.messages if msg["timestamp"] > cutoff]
        pruned_count = original_count - len(self.messages)

        if pruned_count > 0:
            print(f" Pruned {pruned_count} old message(s) due to time expiration.")

    def get_history(self):
        """Return formatted history for the LLM."""
        history = ""
        for msg in self.messages:
            history += f"{msg['role'].capitalize()}: {msg['content']}\n"
        return history

# Demo
print("=== Time-based Expiration Memory Demo (max age = 5 seconds for demo) ===\n")

time_memory = TimeBasedConversationMemory(max_age_minutes=0.1)  # 6 seconds for quick demo

time_memory.add_message("human", "Hi, I'm researching electric car companies. Tell me about Tesla.")#This message should be pruned out
time_memory.add_message("ai", "Tesla is a leading EV company founded by Elon Musk...")

print("Current History:")
print(time_memory.get_history())

print("\nWaiting 8 seconds to trigger pruning...")
import time
time.sleep(8)

time_memory.add_message("human", "How does Tesla compare to Toyota?")

print("\nAfter pruning:")
print(time_memory.get_history())

=== Time-based Expiration Memory Demo (max age = 5 seconds for demo) ===

Current History:
Human: Hi, I'm researching electric car companies. Tell me about Tesla.
Ai: Tesla is a leading EV company founded by Elon Musk...


Waiting 8 seconds to trigger pruning...
 Pruned 2 old message(s) due to time expiration.

After pruning:
Human: How does Tesla compare to Toyota?



## Relevance-based Pruning Memory

Instead of keeping messages based on time or fixed count, we keep the **most relevant** messages using embeddings.

This is widely used in production agents to maintain context without exceeding token limits.

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

class RelevanceBasedMemory:
    def __init__(self, max_messages: int = 6):
        self.max_messages = max_messages
        self.embeddings = OpenAIEmbeddings()
        self.vectorstore = None #vectore for all interactions
        self.messages = [] #temporary store ....======>>>>>>20 rounds === 40 messages in round #21  RAG (documents+vector DBS) + Rag ( last relevant 5 messages)

    def add_message(self, role: str, content: str):
        """Add message and maintain relevance."""
        self.messages.append({"role": role, "content": content})

        doc = Document(
            page_content=content,
            metadata={"role": role, "timestamp": datetime.now()}
        )

        if self.vectorstore is None:
            self.vectorstore = FAISS.from_documents([doc], self.embeddings)
        else:
            self.vectorstore.add_documents([doc])

        # Prune to keep only most relevant messages
        if len(self.messages) > self.max_messages:
            self._prune_to_relevant()

    def _prune_to_relevant(self):
        """Keep only the most relevant messages (simple strategy for demo)."""
        self.messages = self.messages[-self.max_messages:]  # fallback: keep recent

    def get_relevant_history(self, current_query: str, k: int = 5):
        """Retrieve most relevant past messages for current query."""
        if self.vectorstore is None:
            return ""

        docs = self.vectorstore.similarity_search(current_query, k=k)

        history = ""
        for doc in docs:
            history += f"{doc.metadata['role'].capitalize()}: {doc.page_content}\n"
        return history

# Demo
print("=== Relevance-based Memory Demo ===\n")

rel_memory = RelevanceBasedMemory(max_messages=6)

rel_memory.add_message("human", "Hi, I'm researching electric car companies.")
rel_memory.add_message("ai", "Tesla is a leader in EVs with strong growth.")
rel_memory.add_message("human", "How does Tesla compare to Toyota?")
rel_memory.add_message("ai", "Tesla is pure EV, Toyota is hybrid-focused.")

print("Retrieved relevant history for query: 'long-term growth potential'")
print(rel_memory.get_relevant_history("long-term growth potential"))

=== Relevance-based Memory Demo ===

Retrieved relevant history for query: 'long-term growth potential'
Ai: Tesla is a leader in EVs with strong growth.
Human: How does Tesla compare to Toyota?
Human: Hi, I'm researching electric car companies.
Ai: Tesla is pure EV, Toyota is hybrid-focused.



## Task: Build a Memory-Powered Stock Research Conversation Using Buffer Memory

Your task is to build a 4-turn conversation using `ConversationBufferMemory` with
the same Automotive Stock Research theme. The agent must remember context across
all turns without you repeating information.


In [16]:
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import random

# TOOLS
# Using mock data so no API key is needed beyond OpenAI

MOCK_STOCK_DATA = {
    "tesla": {
        "price": 177.90,
        "change_pct": 2.3,
        "market_cap": "565B",
        "pe_ratio": 47.2,
        "52w_high": 299.29,
        "52w_low": 138.80,
    },
    "toyota": {
        "price": 192.45,
        "change_pct": -0.5,
        "market_cap": "267B",
        "pe_ratio": 9.1,
        "52w_high": 226.89,
        "52w_low": 168.22,
    },
    "bmw": {
        "price": 34.12,
        "change_pct": 0.8,
        "market_cap": "51B",
        "pe_ratio": 4.3,
        "52w_high": 42.10,
        "52w_low": 28.90,
    },
    "rivian": {
        "price": 12.87,
        "change_pct": -1.2,
        "market_cap": "13B",
        "pe_ratio": None,
        "52w_high": 24.87,
        "52w_low": 8.26,
    },
    "ford": {
        "price": 11.22,
        "change_pct": 1.1,
        "market_cap": "44B",
        "pe_ratio": 12.4,
        "52w_high": 14.85,
        "52w_low": 9.50,
    },
}

MOCK_NEWS = {
    "tesla": [
        "Tesla delivers record 1.8M vehicles in 2024 despite price cuts.",
        "Elon Musk announces new affordable Tesla model under $30K.",
        "Tesla Autopilot under regulatory scrutiny in the EU.",
    ],
    "toyota": [
        "Toyota doubles down on hybrid strategy, delays full EV switch.",
        "Toyota posts record profit driven by weak yen and strong demand.",
        "Toyota invests $8B in North Carolina battery plant.",
    ],
    "bmw": [
        "BMW sells 375K EVs in 2024, up 34% year-on-year.",
        "BMW i7 wins luxury car of the year award in Europe.",
        "BMW cuts EV production target citing weak European demand.",
    ],
    "rivian": [
        "Rivian narrows losses as Amazon delivery van orders ramp up.",
        "Rivian and Volkswagen joint venture officially announced.",
        "Rivian struggles with supply chain issues at Normal, IL plant.",
    ],
    "ford": [
        "Ford's EV division loses $1.3B in Q1 2024.",
        "Ford Maverick hybrid truck waitlist exceeds 100K customers.",
        "Ford delays next-gen EV pickup launch to 2027.",
    ],
}


@tool
def search_stock_data(company: str) -> str:
    """
    Fetches current stock data for an automotive company.
    Input should be the company name e.g. Tesla, Toyota, BMW, Rivian, Ford.
    """
    key  = company.lower().strip()
    data = MOCK_STOCK_DATA.get(key)

    if not data:
        available = ", ".join(MOCK_STOCK_DATA.keys())
        return f"No data found for '{company}'. Available companies: {available}"

    pe = data["pe_ratio"] if data["pe_ratio"] else "N/A (not yet profitable)"

    return (
        f"{company.title()} Stock Data:\n"
        f"  Current Price : ${data['price']}\n"
        f"  Daily Change  : {data['change_pct']:+.1f}%\n"
        f"  Market Cap    : ${data['market_cap']}\n"
        f"  P/E Ratio     : {pe}\n"
        f"  52-Week High  : ${data['52w_high']}\n"
        f"  52-Week Low   : ${data['52w_low']}\n"
    )


@tool
def get_news_headlines(company: str) -> str:
    """
    Fetches recent news headlines for an automotive company.
    Input should be the company name e.g. Tesla, Toyota, BMW, Rivian, Ford.
    """
    key       = company.lower().strip()
    headlines = MOCK_NEWS.get(key)

    if not headlines:
        available = ", ".join(MOCK_NEWS.keys())
        return f"No news found for '{company}'. Available companies: {available}"

    formatted = "\n".join(f"  - {h}" for h in headlines)
    return f"Recent news for {company.title()}:\n{formatted}"


# MEMORY

memory = ConversationBufferMemory(
    memory_key      = "chat_history",
    return_messages = True,   # must be True for agent — needs message objects
)


# PROMPT
# MessagesPlaceholder injects the full buffer memory into the prompt

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert Automotive Stock Research Agent.
You help investors research car companies by fetching live stock data and news.
You remember everything discussed in the conversation and use that context
to give informed, connected answers.
Never ask the user to repeat information they already gave you."""
    ),
    MessagesPlaceholder(variable_name="chat_history"),   # buffer memory goes here
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


# AGENT

llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
tools = [search_stock_data, get_news_headlines]

agent = create_openai_functions_agent(
    llm    = llm,
    tools  = tools,
    prompt = prompt,
)

agent_executor = AgentExecutor(
    agent   = agent,
    tools   = tools,
    memory  = memory,
    verbose = True,    # set to False to hide tool call details
)


# CONVERSATION — watch how memory carries context across turns

def chat(question: str):
    print(f"Human: {question}")
    response = agent_executor.invoke({"input": question})
    print(f"\nAgent: {response['output']}")
    return response["output"]


print(" Automotive Stock Research Agent with Buffer Memory\n")

# Turn 1 — ask about Tesla
chat("What is Tesla's current stock price and any recent news about them?")

# Turn 2 — ask about Toyota
chat("Now look up Toyota for me — price and news please.")

# Turn 3 — comparison, NO company names repeated
# The agent must use memory to know we discussed Tesla and Toyota
chat("Based on the two companies we just discussed, which one looks more stable right now?")

# Turn 4 — follow-up using earlier context
# The agent must remember the news and prices it already fetched
chat("Which of the risks you mentioned earlier concerns you most for a 2-year investment horizon?")


# INSPECT MEMORY


print("\n\n Buffer Memory Contents After Conversation \n")
for i, msg in enumerate(memory.chat_memory.messages, 1):
    role    = "Human" if msg.type == "human" else "Agent"
    preview = msg.content[:120].replace("\n", " ")
    print(f"[{i:02d}] {role:6s}: {preview}...")

print(f"\nTotal messages in buffer : {len(memory.chat_memory.messages)}")
print(f"Total characters stored  : {sum(len(m.content) for m in memory.chat_memory.messages)}")
print(f"Estimated tokens         : ~{sum(len(m.content) for m in memory.chat_memory.messages) // 4}")

 Automotive Stock Research Agent with Buffer Memory

Human: What is Tesla's current stock price and any recent news about them?


> Entering new AgentExecutor chain...

Invoking: `search_stock_data` with `{'company': 'Tesla'}`


Tesla Stock Data:
  Current Price : $177.9
  Daily Change  : +2.3%
  Market Cap    : $565B
  P/E Ratio     : 47.2
  52-Week High  : $299.29
  52-Week Low   : $138.8

Invoking: `get_news_headlines` with `{'company': 'Tesla'}`


Recent news for Tesla:
  - Tesla delivers record 1.8M vehicles in 2024 despite price cuts.
  - Elon Musk announces new affordable Tesla model under $30K.
  - Tesla Autopilot under regulatory scrutiny in the EU.Tesla's current stock price is $177.90, with a daily change of +2.3%. The company's market cap is $565 billion, and it has a P/E ratio of 47.2. The stock has a 52-week high of $299.29 and a low of $138.80.

### Recent News:
1. Tesla delivers a record 1.8 million vehicles in 2024 despite price cuts.
2. Elon Musk announces a new affor

## Conclusion

**Key Takeaways:**

- Conversation memory is essential for multi-turn interactions.
- Short-term buffer memory is simple but grows fast.
- Window memory helps control context length.
- Summary memory reduces token usage while preserving meaning.
- In real production agents, a hybrid approach (window + summary + vector retrieval) is commonly used.